<a href="https://colab.research.google.com/github/Mustafizb24/gold-directional-forecasting/blob/main/2_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Connecting Colab to drive
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

#Load the features dataset saved in notebook 01, using dates as the index
df = pd.read_csv("/content/drive/MyDrive/gold_features.csv", index_col=0, parse_dates=True)

# Checking rows and columns to verify
print(df.shape)

Mounted at /content/drive
(3974, 12)


In [2]:
from sklearn.model_selection import TimeSeriesSplit #Main tool for walk-forward splits

# The 8 features used as model inputs (raw prices are left out)
feature_cols = ["gold_ret", "dxy_ret", "yield_chg", "gold_ret_lag1", "gold_ret_lag2", "gold_ma20_gap", "gold_ma50_gap", "gold_vol20"]

# x = inputs, y = what we're predicting
x = df[feature_cols]
y = df["target"]

#Walk-forward: 5 splits, each one trains on all past data and tests on the next block of time
tscv = TimeSeriesSplit(n_splits = 5)


In [3]:
# Show the training and testing dates for each split
for i, (train_idx, test_idx) in enumerate(tscv.split(x), start=1):
    train_dates = x.index[train_idx]
    test_dates = x.index[test_idx]

    print(f"Split {i}:")
    print("Train:", train_dates.min().date(), "to", train_dates.max().date(), "-", len(train_idx), "rows")
    print("Test:", test_dates.min().date(), "to", test_dates.max().date(), "-", len(test_idx), "rows")
    print()

Split 1:
Train: 2010-03-16 to 2012-10-31 - 664 rows
Test: 2012-11-01 to 2015-06-19 - 662 rows

Split 2:
Train: 2010-03-16 to 2015-06-19 - 1326 rows
Test: 2015-06-22 to 2018-02-08 - 662 rows

Split 3:
Train: 2010-03-16 to 2018-02-08 - 1988 rows
Test: 2018-02-09 to 2020-09-25 - 662 rows

Split 4:
Train: 2010-03-16 to 2020-09-25 - 2650 rows
Test: 2020-09-28 to 2023-05-15 - 662 rows

Split 5:
Train: 2010-03-16 to 2023-05-15 - 3312 rows
Test: 2023-05-16 to 2025-12-30 - 662 rows



In [4]:
from sklearn.metrics import accuracy_score, f1_score

# Random walk: predict tomorrow moves the same way as today (1 if today's return was up)
rw_pred = (df["gold_ret"] > 0).astype(int)

rw_results = []

# Scores the random walk on each test period
for i, (train_idx, test_idx) in enumerate(tscv.split(x), start=1):
    y_test = y.iloc[test_idx]
    pred = rw_pred.iloc[test_idx]

    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, zero_division=0)

    rw_results.append([i, acc, f1])
    print("Split", i, "- Accuracy:", round(acc, 3), "F1:", round(f1, 3))

Split 1 - Accuracy: 0.444 F1: 0.412
Split 2 - Accuracy: 0.511 F1: 0.509
Split 3 - Accuracy: 0.458 F1: 0.485
Split 4 - Accuracy: 0.476 F1: 0.509
Split 5 - Accuracy: 0.502 F1: 0.555


In [6]:
majority_results = []

# Majority class baseline: it predicts the most common direction from training data everyday
for i, (train_idx, test_idx) in enumerate(tscv.split(x), start=1):
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    majority = y_train.mode()[0] # The mode returns the most common value (1 = up, 0 = down)
    pred = [majority] * len(y_test) #Same guess for every test day

    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, zero_division=0)

    majority_results.append([i, acc, f1])
    print("Split", i, "- Majority:", majority ,"Accuracy",round(acc, 3), "F1:", round(f1, 3))

Split 1 - Majority: 1 Accuracy 0.473 F1: 0.642
Split 2 - Majority: 1 Accuracy 0.498 F1: 0.665
Split 3 - Majority: 1 Accuracy 0.527 F1: 0.69
Split 4 - Majority: 1 Accuracy 0.533 F1: 0.696
Split 5 - Majority: 1 Accuracy 0.56 F1: 0.718


In [8]:
# Turn each baseline's results into a table
rw_df = pd.DataFrame(rw_results, columns=["split", "accuracy", "f1"])
rw_df["model"] = "Random Walk"

maj_df = pd.DataFrame(majority_results, columns=["split", "accuracy", "f1"])
maj_df["model"] = "Majority class"

# Stack both tables into one
results = pd.concat([rw_df, maj_df])
print(results)

#Average score of each baseline across the 5 splits
print(results.groupby("model")[["accuracy","f1"]].mean().round(3))

#Save to Drive so other notebooks can compare models against these
results.to_csv("/content/drive/MyDrive/baseline_results.csv", index = False)

   split  accuracy        f1           model
0      1  0.444109  0.412141     Random Walk
1      2  0.510574  0.509091     Random Walk
2      3  0.457704  0.484935     Random Walk
3      4  0.475831  0.509194     Random Walk
4      5  0.501511  0.555256     Random Walk
0      1  0.472810  0.642051  Majority class
1      2  0.498489  0.665323  Majority class
2      3  0.527190  0.690406  Majority class
3      4  0.533233  0.695567  Majority class
4      5  0.560423  0.718296  Majority class
                accuracy     f1
model                          
Majority class     0.518  0.682
Random Walk        0.478  0.494
